# JeevaSwara / KanthaRakshak: 03. Feature Extraction & Spectral Analysis

This notebook analyzes the 25+ biomechanical features extracted from throat acoustics and cervical kinematics, including:
- FFT Spectral Moments (Centroid, Bandwidth, Rolloff, Peaks)
- 13 Mel-Frequency Cepstral Coefficients (MFCCs)
- Cross-sensor lag and correlation


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from app.pipeline.model_trainer import NUMERICAL_FEATURE_COLUMNS

df = pd.read_csv("../data/research_cohort_sessions.csv")
print(f"Analyzing {len(NUMERICAL_FEATURE_COLUMNS)} numerical features across {len(df)} sessions.")


### Feature Distributions: Normal vs Abnormal Swallows
Let us compare duration, dominant frequency, jerk RMS, and sensor agreement score between the two clinical groups.


In [ ]:
features_to_plot = [
    ("swallow_duration_ms", "Swallow Duration (ms)", [400, 2400]),
    ("dominant_frequency_hz", "Dominant Frequency (Hz)", [4, 20]),
    ("spectral_peak_count", "Spectral Energy Bursts", [0, 8]),
    ("piezo_motion_peak_delay_ms", "Piezo-Motion Lag (ms)", [0, 350])
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, (col, title, xlim) in enumerate(features_to_plot):
    ax = axes[idx]
    normal_vals = df[df['label'] == 'NORMAL'][col]
    abnormal_vals = df[df['label'] == 'ABNORMAL'][col]
    
    ax.hist(normal_vals, bins=15, alpha=0.6, label='NORMAL', color='#10b981', density=True)
    ax.hist(abnormal_vals, bins=15, alpha=0.6, label='ABNORMAL', color='#f43f5e', density=True)
    ax.set_title(title)
    ax.set_xlim(xlim)
    ax.legend()

plt.tight_layout()
plt.show()


### Feature Correlation Matrix
Checking for multicollinearity and informative feature clusters.


In [ ]:
core_cols = [
    "swallow_duration_ms", "dominant_frequency_hz", "spectral_peak_count",
    "max_acceleration_g", "jerk_rms_g_per_s", "piezo_motion_peak_delay_ms",
    "cross_correlation_coeff", "sensor_agreement_score"
]

corr = df[core_cols].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Pearson Correlation')
plt.xticks(range(len(core_cols)), core_cols, rotation=45, ha='right')
plt.yticks(range(len(core_cols)), core_cols)
plt.title('Feature Correlation Heatmap')
for i in range(len(core_cols)):
    for j in range(len(core_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)
plt.tight_layout()
plt.show()
